# SAR-MNL: Spatial Autoregressive Multinomial Logit Demo

This notebook demonstrates the `SARMNL` class — a spatial autoregressive MNL model
that specifies a spatial lag in the systematic utility of alternatives (locations).

The model specifies:

$$V_j = \rho \sum_k w_{jk} V_k + Z_j \beta + X_{ij} \gamma$$

yielding reduced-form utilities $V^* = (I - \rho W)^{-1}(Z\beta + X\gamma)$,
normalised by $D = \text{diag}((I - \rho W)^{-1})$, with standard MNL choice probabilities.

Estimation is via pseudo maximum likelihood (PML, Smirnov 2010) with JAX autodiff.

## 1. Setup and Imports

In [ ]:
import time

import numpy as np
import pandas as pd

from locpick import ChoiceModel
from locpick.dgp import simulate_sar_mnl

In [ ]:
import geosnap as gsp

In [ ]:
datasets = gsp.DataStore()

In [ ]:
dc = gsp.io.get_acs(datasets, years=2019, level="tract", state_fips="11")

In [ ]:
dc.plot()

In [ ]:
from libpysal.graph import Graph

dc_graph = Graph.build_contiguity(dc, rook=False)

adj = dc_graph.sparse.todense()

## 2. Data Preparation

We generate synthetic data from a known SAR-MNL data generating process using `simulate_sar_mnl`. This creates:
- **Choosers**: observations with an income attribute
- **Alternatives**: spatial locations with cost and time attributes
- **Spatial weights matrix (W)**: a circular adjacency graph connecting nearby alternatives
- **Choices**: simulated from the SAR-MNL probability with a known spatial autoregressive parameter ρ

In [ ]:
# Generate synthetic SAR-MNL data
# n_obs=2000 choosers, n_alts=12 alternatives, rho=0.5
dataset = simulate_sar_mnl(n_obs=2000, n_alts=dc.shape[0], rho=0.5, seed=42, W=dc_graph)

print(f"Observations: {dataset.n_obs}")
print(f"Alternatives: {dataset.n_alts}")
print(f"True rho: {dataset.true_rho}")
print(f"True params: {dataset.true_params}")
print(f"\nChoosers columns: {list(dataset.choosers.columns)}")
print(f"Alternatives columns: {list(dataset.alternatives.columns)}")
print(f"\nChoice table: {dataset.choice_table}")

## 3. Instantiating the SARMNL Class

The `SARMNL` class takes the same data interface as `ChoiceModel`, plus a spatial weights matrix `W`. The weights matrix defines the spatial relationships between alternatives (locations).

Key parameters:
- **`data`**: A `ChoiceTable` containing choosers, alternatives, and choices
- **`formula`**: A formulaic formula string for the utility function
- **`W`**: Spatial weights matrix (libpysal Graph, scipy.sparse, or np.ndarray) — row-standardised internally
- **`solver`**: Default `"lbfgs"` (scipy L-BFGS-B, the fastest path)

In [ ]:
dataset.choice_table.to_frame()

In [ ]:
# Create the SAR-MNL model
# W is the spatial weights matrix from the DGP (circular adjacency)
model_sar = ChoiceModel(
    dataset.choice_table, formula="alt_attr + obs_x_alt - 1", graph=dataset.W, lag=True
)

print(f"Model: {model_sar}")
print(f"W shape: {dataset.W.n}")
print(f"W type: {type(dataset.W).__name__}")

## 4. Model Estimation

Fit the SAR-MNL model and compare with a standard MNL (no spatial lag). The spatial autoregressive parameter ρ captures the degree of spatial spillover in utilities — nearby locations influence each other's attractiveness.

In [ ]:
# Fit SAR-MNL
t0 = time.perf_counter()
result_sar = model_sar.fit()
t_sar = time.perf_counter() - t0

print("=== SAR-MNL Results ===")
print(result_sar.summary())
print(f"\nFit time: {t_sar:.2f}s")

In [ ]:
# Fit standard MNL (no spatial lag) for comparison
model_mnl = ChoiceModel(dataset.choice_table, formula="alt_attr + obs_x_alt - 1")
t0 = time.perf_counter()
result_mnl = model_mnl.fit()
t_mnl = time.perf_counter() - t0

print("=== Standard MNL Results ===")
print(result_mnl.summary())
print(f"\nFit time: {t_mnl:.2f}s")

## 5. Results Visualization

Compare the estimated parameters from SAR-MNL vs standard MNL, and check how well the SAR-MNL model recovers the true spatial autoregressive parameter ρ.

In [ ]:
# Compare coefficient estimates
print("=== Parameter Recovery ===")
print(f"True rho: {dataset.true_rho:.3f}")
print(f"Estimated rho (SAR-MNL): {result_sar.coefficients['rho']:.3f}")
print()

# Side-by-side coefficient comparison
comparison = pd.DataFrame(
    {
        "True": [dataset.true_params.get(c, np.nan) for c in result_sar.coefficients.index],
        "SAR-MNL": result_sar.coefficients.values,
        "MNL": [result_mnl.coefficients.get(c, np.nan) for c in result_sar.coefficients.index],
    },
    index=result_sar.coefficients.index,
)
print(comparison.round(4))

In [ ]:
# Fit statistics comparison
print("=== Fit Statistics ===")
stats_df = pd.DataFrame(
    {
        "SAR-MNL": [
            result_sar.log_likelihood,
            result_sar.aic,
            result_sar.bic,
            result_sar.rho_squared,
            result_sar.rho_bar_squared,
        ],
        "MNL": [
            result_mnl.log_likelihood,
            result_mnl.aic,
            result_mnl.bic,
            result_mnl.rho_squared,
            result_mnl.rho_bar_squared,
        ],
    },
    index=["Log-likelihood", "AIC", "BIC", "rho^2", "rho-bar^2"],
)
print(stats_df.round(4))

In [ ]:
# Visualize spatial spillover: how does rho affect utilities?
# Compare raw utilities vs spatially-filtered utilities
probs_sar = model_sar.probabilities()
probs_mnl = model_mnl.probabilities()

print("=== Probability Comparison (first 5 obs, first 5 alts) ===")
print("SAR-MNL probabilities:")
print(probs_sar[:5, :5].round(4))
print("\nMNL probabilities:")
print(probs_mnl[:5, :5].round(4))

# The spatial autoregressive structure redistributes probability mass
# toward alternatives that are surrounded by attractive neighbors
print(f"\nMean absolute probability difference: {np.mean(np.abs(probs_sar - probs_mnl)):.6f}")

In [ ]:
dc.assign(probs=probs_sar[0]).plot("probs", scheme="quantiles")

## 6. Performance Benchmarking

Benchmark SAR-MNL vs standard MNL to highlight the computational cost of the spatial solve (matrix inversion at each evaluation). The hybrid estimation path (scipy LBFGS + JAX kernels + JAX autodiff) is used for both models.

In [ ]:
# Warm fit (second fit, post-JIT cache)
t0 = time.perf_counter()
result_sar_warm = model_sar.fit()
t_sar_warm = time.perf_counter() - t0

t0 = time.perf_counter()
result_mnl_warm = model_mnl.fit()
t_mnl_warm = time.perf_counter() - t0

print("=== Warm Fit Performance ===")
print(f"SAR-MNL: {t_sar_warm:.3f}s (cold: {t_sar:.3f}s)")
print(f"MNL:     {t_mnl_warm:.3f}s (cold: {t_mnl:.3f}s)")
print(f"SAR overhead: {t_sar_warm / t_mnl_warm:.1f}x")
print("\nBoth use the hybrid path: scipy LBFGS + JAX JIT'd kernels + JAX autodiff gradients")

## Summary

The `ChoiceModel` class provides spatial autoregressive MNL estimation via pseudo maximum likelihood:

- **Spatial weights**: Accepts libpysal Graph, scipy.sparse, or dense ndarray for `W`
- **JAX-accelerated**: Log-likelihood and gradient computed via JAX autodiff through the spatial solve
- **Hybrid estimation**: scipy LBFGS solver + JAX kernels (the fastest path for all locpick models)
- **Parameter recovery**: The spatial autoregressive parameter ρ is estimated alongside utility coefficients

### When to use SAR-MNL vs SCL

| Feature | SAR-MNL | SCL (ChoiceModel + graph) |
|---|---|---|
| Spatial structure | Autoregressive lag in utility | Paired GEV nests |
| Estimation | Pseudo-ML (no Jacobian) | Full-likelihood (GEV) |
| ρ range | (-1, 1) | (0, 1] |
| Variance normalisation | diag((I-ρW)⁻¹) | None (GEV handles it) |
| Computational cost | Matrix solve per eval | Scatter operations |

Use **SAR-MNL** when you want a spatial lag in utilities (spillover effects).
Use **SCL** when you want spatial correlation in the error structure (GEV).